# This ipynb answers RQ2: How do frames co-occur in the same excerpt, and what can we learn about meaningful foregrounding and backgrounding from differences in binary variables? 

In [1]:
import pandas as pd

merged_df = pd.read_csv('full_data.csv')
merged_df.head(1)

,row_id,text-topic,text-topic-exp,text-entity-name,text-entity-gender,text-entity-sentiment,text-entity-sentiment-exp,text-generic-frame,text-generic-frame-exp,text-issue-frame,...,excerpt_length,gpt_title_frame,gpt_excerpt_primary_frame,gpt_excerpt_secondary_frame,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present,frames_error,binary_error
0,1,Ethiopia's Political Instability and State of ...,The article discusses the declaration of a sta...,Ethiopia's Amhara Region,NaN,Negative,The article discusses a state of emergency bei...,"['Security and defense', 'Economic', 'Policy p...",The article discusses a state of emergency dec...,Political Instability and Security Concerns,...,1321,residual,disruption,harm_to_protesters,0,0,1,1,NaN,NaN


In [2]:
# Keep only observations with a secondary frame
secondary_df = merged_df[
    merged_df["gpt_excerpt_secondary_frame"].notna()
].copy()

primary_secondary_pct = (
    pd.crosstab(
        secondary_df["gpt_excerpt_primary_frame"],
        secondary_df["gpt_excerpt_secondary_frame"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

primary_secondary_pct

gpt_excerpt_secondary_frame,disruption,grievance,harm_to_protesters,impactful,peaceful,property_damage,protester_violence,widespread_support
gpt_excerpt_primary_frame,,,,,,,,
disruption,0.0,60.4,5.5,13.3,2.8,3.8,8.9,5.2
grievance,26.7,0.0,10.3,12.1,8.8,1.9,3.5,36.7
harm_to_protesters,12.1,44.6,0.0,9.1,12.9,2.1,13.1,6.1
impactful,26.9,38.4,7.6,0.0,0.7,2.4,9.1,15.0
peaceful,11.9,46.8,8.3,2.8,0.0,1.8,11.0,17.4
property_damage,26.7,22.6,5.7,6.0,1.9,0.0,34.9,2.2
protester_violence,44.6,10.7,11.0,10.5,1.9,18.9,0.0,2.5
widespread_support,6.9,72.4,4.5,6.9,5.1,0.3,3.9,0.0


Cues for these are often both substantively in the same article:
- Disruption and Grievance 
- Grievance and Widespread_Support
- protester violence and property damage
- impactful and disruption (makes sense for boycotts)

In [3]:
secondary_rate_by_primary = (
    merged_df
    .assign(
        has_secondary=merged_df["gpt_excerpt_secondary_frame"].notna()
    )
    .groupby("gpt_excerpt_primary_frame")["has_secondary"]
    .agg(["mean", "sum", "count"])
)

secondary_rate_by_primary["secondary_rate"] = (
    secondary_rate_by_primary["mean"] * 100
).round(1)

secondary_rate_by_primary = secondary_rate_by_primary[
    ["count", "sum", "secondary_rate"]
].rename(columns={
    "count": "N",
    "sum": "N_with_secondary"
})

secondary_rate_by_primary

,N,N_with_secondary,secondary_rate
gpt_excerpt_primary_frame,,,
disruption,1658,1265,76.3
grievance,2441,1384,56.7
harm_to_protesters,1145,816,71.3
impactful,782,594,76.0
peaceful,154,109,70.8
property_damage,403,318,78.9
protester_violence,1907,1183,62.0
residual,350,0,0.0
widespread_support,443,333,75.2


In [ ]:
import pandas as pd
import numpy as np

primary_col = "gpt_excerpt_primary_frame"
secondary_col = "gpt_excerpt_secondary_frame"

# Normalize labels to avoid capitalization inconsistencies
df_pairs = merged_df.copy()

for col in [primary_col, secondary_col]:
    df_pairs[col] = (
        df_pairs[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

# pairs of interest
frame_pairs = [
    ("disruption", "grievance"),
    ("grievance", "protester_violence"),
    ("grievance", "property_damage"), 
    ("harm_to_protesters", "disruption"),
    ("harm_to_protesters", "protester_violence"),
    ("harm_to_protesters", "property_damage"),
    ("grievance", "widespread_support"),
    ("protester_violence", "property_damage"),
    ("impactful", "disruption")
]

rows = []

for a, b in frame_pairs:
    ab = (
        (df_pairs[primary_col] == a) &
        (df_pairs[secondary_col] == b)
    ).fillna(False)

    ba = (
        (df_pairs[primary_col] == b) &
        (df_pairs[secondary_col] == a)
    ).fillna(False)

    n_ab, n_ba = int(ab.sum()), int(ba.sum())
    total = n_ab + n_ba

    rows.append({
        "Pair": f"{a} / {b}",
        f"{a} → {b}": n_ab,
        f"{b} → {a}": n_ba,
        "Total": total,
        "First ordering (%)": round(100 * n_ab / total, 1) if total else np.nan
    })

pair_counts = pd.DataFrame(rows)
pair_counts

,Pair,disruption → grievance,grievance → disruption,Total,First ordering (%),grievance → protester_violence,protester_violence → grievance,grievance → property_damage,property_damage → grievance,harm_to_protesters → disruption,...,harm_to_protesters → protester_violence,protester_violence → harm_to_protesters,harm_to_protesters → property_damage,property_damage → harm_to_protesters,grievance → widespread_support,widespread_support → grievance,protester_violence → property_damage,property_damage → protester_violence,impactful → disruption,disruption → impactful
0,disruption / grievance,764.0,369.0,1133,67.4,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,grievance / protester_violence,NaN,NaN,174,27.6,48.0,126.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,grievance / property_damage,NaN,NaN,98,26.5,NaN,NaN,26.0,72.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,harm_to_protesters / disruption,NaN,NaN,169,58.6,NaN,NaN,NaN,NaN,99.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,harm_to_protesters / protester_violence,NaN,NaN,237,45.1,NaN,NaN,NaN,NaN,NaN,...,107.0,130.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,harm_to_protesters / property_damage,NaN,NaN,35,48.6,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,17.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN
6,grievance / widespread_support,NaN,NaN,749,67.8,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,508.0,241.0,NaN,NaN,NaN,NaN
7,protester_violence / property_damage,NaN,NaN,334,66.8,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,223.0,111.0,NaN,NaN
8,impactful / disruption,NaN,NaN,328,48.8,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,160.0,168.0


Are there meaningful differences between Disruption-Grievance and Grievance-Disruption? And other pairs

In [15]:
binary_cols = [
    "gpt_demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]

pair_dfs = {}
summary_rows = []

for a, b in frame_pairs:
    temp = df_pairs[
        (
            (df_pairs[primary_col] == a) &
            (df_pairs[secondary_col] == b)
        ) |
        (
            (df_pairs[primary_col] == b) &
            (df_pairs[secondary_col] == a)
        )
    ].copy()

    # 1 means a is primary; 0 means b is primary
    temp["a_primary"] = (temp[primary_col] == a).astype(int)
    temp["ordering"] = np.where(
        temp["a_primary"] == 1,
        f"{a} → {b}",
        f"{b} → {a}"
    )

    pair_dfs[(a, b)] = temp

    for ordering, group in temp.groupby("ordering"):
        row = {
            "Pair": f"{a} / {b}",
            "Ordering": ordering,
            "N": len(group)
        }

        for col in binary_cols:
            row[col] = group[col].mean() * 100

        summary_rows.append(row)

pair_feature_summary = pd.DataFrame(summary_rows)

pair_feature_summary[binary_cols] = (
    pair_feature_summary[binary_cols].round(1)
)

pair_feature_summary

,Pair,Ordering,N,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present
0,disruption / grievance,disruption → grievance,764,99.0,72.6,79.1,59.7
1,disruption / grievance,grievance → disruption,369,99.5,83.5,75.6,69.9
2,grievance / protester_violence,grievance → protester_violence,48,97.9,75.0,66.7,62.5
3,grievance / protester_violence,protester_violence → grievance,126,98.4,69.0,81.7,62.7
4,grievance / property_damage,grievance → property_damage,26,92.3,61.5,80.8,84.6
5,grievance / property_damage,property_damage → grievance,72,98.6,51.4,77.8,80.6
6,harm_to_protesters / disruption,disruption → harm_to_protesters,70,77.1,58.6,82.9,55.7
7,harm_to_protesters / disruption,harm_to_protesters → disruption,99,86.9,65.7,93.9,64.6
8,harm_to_protesters / protester_violence,harm_to_protesters → protester_violence,107,58.9,37.4,89.7,46.7
9,harm_to_protesters / protester_violence,protester_violence → harm_to_protesters,130,71.5,40.8,93.1,61.5


We observe pretty big gaps in protester quote between Pair 1, Pair 2: in both cases, grievance being chosen as primary has the greater rate of protester quotes. This single text feature is likely a big cue for grievance. 

Both directions in Pair 3 (protester_violence and property damage) see very high 90% of authority quotes. These are likely police/sheriff statements or law enforcement. Upholding the status quo strongly

There seems to be some drop off between primary and secondary (i.e. not equal mix ups). From both Pair 1 and Pair 2, we see grievance having much higher protester quotes. From impactful-disruption, impactful as primary sees a 16pp increase in thematic present, which makes sense and is part of what impactful means. 

Interestingly, there is not a clear theoretical, one-to-one connection between the binaries and whether legit/delegit is foregrounded: e.g. harm to protesters -> protester violence as primary has lower demands present, protester quotes, and thematic present than the flip side. 

In [7]:
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

test_rows = []

for (a, b), temp in pair_dfs.items():
    for feature in binary_cols:

        valid = temp[["a_primary", feature]].dropna()

        group_a = valid.loc[valid["a_primary"] == 1, feature]
        group_b = valid.loc[valid["a_primary"] == 0, feature]

        if len(group_a) == 0 or len(group_b) == 0:
            continue

        count = [group_a.sum(), group_b.sum()]
        nobs = [len(group_a), len(group_b)]

        # Two-sided difference-in-proportions test
        z, p = proportions_ztest(count, nobs)

        test_rows.append({
            "Pair": f"{a} / {b}",
            "Feature": feature,
            "N_a": len(group_a),
            "N_b": len(group_b),
            "Rate A (%)": group_a.mean() * 100,
            "Rate B (%)": group_b.mean() * 100,
            "Difference (pp)": (
                group_a.mean() - group_b.mean()
            ) * 100,
            "p-value": p
        })

pair_tests = pd.DataFrame(test_rows)

pair_tests["p-adjusted"] = multipletests(
    pair_tests["p-value"],
    method="fdr_bh"
)[1]

pair_tests.round(3)

,Pair,Feature,N_a,N_b,Rate A (%),Rate B (%),Difference (pp),p-value,p-adjusted
0,disruption / grievance,gpt_demands_present,764,369,98.953,99.458,-0.505,0.394,0.574
1,disruption / grievance,gpt_protester_quote,764,369,72.644,83.469,-10.825,0.000,0.000
2,disruption / grievance,gpt_authority_quote,764,369,79.058,75.610,3.448,0.190,0.337
3,disruption / grievance,gpt_thematic_present,764,369,59.686,69.919,-10.233,0.001,0.004
4,grievance / widespread_support,gpt_demands_present,508,241,98.031,97.510,0.521,0.645,0.794
5,grievance / widespread_support,gpt_protester_quote,508,241,72.441,54.772,17.669,0.000,0.000
6,grievance / widespread_support,gpt_authority_quote,508,241,69.094,71.784,-2.690,0.453,0.604
7,grievance / widespread_support,gpt_thematic_present,508,241,72.441,72.199,0.242,0.945,0.945
8,protester_violence / property_damage,gpt_demands_present,223,111,67.265,72.973,-5.708,0.287,0.460
9,protester_violence / property_damage,gpt_protester_quote,223,111,39.462,29.730,9.732,0.081,0.217


In [ ]:
main_topics = [
    "War",
    "Social Issues",
    "Labor",
    "Legal",
    "Politics",
    "Crime"
]

# Count topics across the full dataset
topic_counts = df_pairs["gpt-topic"].value_counts()

# Identify sparse topics (<10), excluding six main topics
sparse_topics = topic_counts[
    (topic_counts < 10) &
    (~topic_counts.index.isin(main_topics))
].index

# Create grouped topic variable
df_pairs["topic_grouped"] = df_pairs["gpt-topic"].replace(
    {topic: "Other" for topic in sparse_topics}
)

df_pairs["topic_grouped"].value_counts()

In [9]:
for pair, temp in pair_dfs.items():
    pair_dfs[pair]["topic_grouped"] = df_pairs.loc[
        temp.index, "topic_grouped"
    ]

Logistic regression. Among articles with both of the frames, what is P(Disruption is primary | Disruption, Grievance in it)? 

In [11]:
import statsmodels.formula.api as smf

temp = pair_dfs[("disruption", "grievance")].copy()

LEAN = 'C(political_leaning, Treatment(reference="center"))'

# Use your existing topic_grouped if available
# Otherwise use gpt-topic categories or create topic_grouped first
TOPIC = 'C(topic_grouped, Treatment(reference="War"))'

# Nested models
formulas = {
    "M1: Textual features": (
        "a_primary ~ "
        "gpt_protester_quote + gpt_authority_quote + "
        "gpt_thematic_present"
    ),
    "M2: + Topic + Lean": (
        "a_primary ~ "
        "gpt_protester_quote + gpt_authority_quote + "
        "gpt_thematic_present + "
        f"{TOPIC} + {LEAN}"
    )
}

# Use identical observations across models
model_df = temp.dropna(
    subset=[
        "a_primary",
        "gpt_protester_quote",
        "gpt_authority_quote",
        "gpt_thematic_present",
        "topic_grouped",
        "political_leaning"
    ]
).copy()

models = {}

for name, formula in formulas.items():
    models[name] = smf.logit(
        formula,
        data=model_df
    ).fit(disp=False)

for name, model in models.items():
    print(f"\n{name}")
    display(pd.DataFrame({
        "OR": np.exp(model.params),
        "p-value": model.pvalues
    }).round(3))


M1: Textual features


/Users/jeremy/Documents/protests_chai/venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


,OR,p-value
Intercept,3.923,0.000
gpt_protester_quote,0.531,0.000
gpt_authority_quote,1.204,0.223
gpt_thematic_present,0.629,0.001



M2: + Topic + Lean


,OR,p-value
Intercept,5.015000e+00,0.000
"C(topic_grouped, Treatment(reference=""War""))[T.Business]",6.260000e-01,0.326
"C(topic_grouped, Treatment(reference=""War""))[T.Crime]",8.970000e-01,0.861
"C(topic_grouped, Treatment(reference=""War""))[T.Culture]",6.430000e-01,0.617
"C(topic_grouped, Treatment(reference=""War""))[T.Economy]",6.590000e-01,0.634
"C(topic_grouped, Treatment(reference=""War""))[T.Education]",2.501000e+00,0.088
"C(topic_grouped, Treatment(reference=""War""))[T.Entertainment]",3.490000e-01,0.019
"C(topic_grouped, Treatment(reference=""War""))[T.Environment]",2.439000e+00,0.013
"C(topic_grouped, Treatment(reference=""War""))[T.Health]",7.319767e+08,0.999
"C(topic_grouped, Treatment(reference=""War""))[T.Housing]",1.653488e+08,0.999


After controlling for topic + lean, thematic present and protester quote both have ~0.5 odds ratio, indicating that their presence makes it stat. significantly less likely to be Disruption rather than Grievance. 

# Selective Salience: Do certain political leanings foreground one frame over the other, within a pair?

In [16]:
lean_order = [
    "left", "left_lean", "center", "right_lean", "right"
]

rows = []

for (a, b), temp in pair_dfs.items():

    summary = (
        temp.groupby("political_leaning")["a_primary"]
        .agg(["mean", "sum", "count"])
        .reindex(lean_order)
    )

    for lean, row in summary.iterrows():
        rows.append({
            "Pair": f"{a} / {b}",
            "Political Leaning": lean,
            "N": row["count"],
            f"{a} primary (%)": row["mean"] * 100
        })

lean_pair_summary = pd.DataFrame(rows)

# Use a consistent column name across all pairs
lean_pair_summary = lean_pair_summary.rename(
    columns={
        col: "First frame primary (%)"
        for col in lean_pair_summary.columns
        if col.endswith("primary (%)")
    }
)

lean_pair_summary = lean_pair_summary.round(1)

lean_pair_summary

,Pair,Political Leaning,N,First frame primary (%),First frame primary (%),First frame primary (%),First frame primary (%),First frame primary (%)
0,disruption / grievance,left,46.0,39.1,NaN,NaN,NaN,NaN
1,disruption / grievance,left_lean,531.0,61.6,NaN,NaN,NaN,NaN
2,disruption / grievance,center,112.0,72.3,NaN,NaN,NaN,NaN
3,disruption / grievance,right_lean,113.0,69.0,NaN,NaN,NaN,NaN
4,disruption / grievance,right,331.0,78.5,NaN,NaN,NaN,NaN
5,grievance / protester_violence,left,8.0,NaN,25.0,NaN,NaN,NaN
6,grievance / protester_violence,left_lean,75.0,NaN,36.0,NaN,NaN,NaN
7,grievance / protester_violence,center,12.0,NaN,33.3,NaN,NaN,NaN
8,grievance / protester_violence,right_lean,15.0,NaN,33.3,NaN,NaN,NaN
9,grievance / protester_violence,right,64.0,NaN,15.6,NaN,NaN,NaN


In [17]:
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

test_rows = []

for (a, b), temp in pair_dfs.items():

    ct = pd.crosstab(
        temp["political_leaning"],
        temp["a_primary"]
    ).reindex(
        index=lean_order,
        columns=[0, 1],
        fill_value=0
    )

    # Remove empty leaning categories
    ct = ct.loc[ct.sum(axis=1) > 0]

    if ct.shape[0] < 2 or (ct.sum(axis=0) == 0).any():
        continue

    chi2, p, dof, expected = chi2_contingency(ct)

    test_rows.append({
        "Pair": f"{a} / {b}",
        "N": ct.to_numpy().sum(),
        "Chi-square": chi2,
        "df": dof,
        "p-value": p,
        "Min expected count": expected.min()
    })

lean_pair_tests = pd.DataFrame(test_rows)

lean_pair_tests["p-adjusted"] = multipletests(
    lean_pair_tests["p-value"],
    method="fdr_bh"
)[1]

lean_pair_tests.round(4)

,Pair,N,Chi-square,df,p-value,Min expected count,p-adjusted
0,disruption / grievance,1133,45.0319,4,0.0000,14.9815,0.0000
1,grievance / protester_violence,174,7.7148,4,0.1026,2.2069,0.2309
2,grievance / property_damage,98,0.4774,4,0.9757,0.7959,0.9757
3,harm_to_protesters / disruption,169,2.0183,4,0.7324,5.3846,0.9416
4,harm_to_protesters / protester_violence,237,0.5925,4,0.9639,5.8692,0.9757
5,harm_to_protesters / property_damage,35,2.3066,4,0.6796,0.9714,0.9416
6,grievance / widespread_support,749,5.4419,4,0.2449,14.8011,0.4408
7,protester_violence / property_damage,334,8.4614,4,0.0761,5.3174,0.2282
8,impactful / disruption,328,8.5159,4,0.0744,5.8537,0.2282


Among Jan 6 articles, do we see differences among political leanings?

In [25]:
import pandas as pd
jan6_df = pd.read_csv('jan6_articles.csv')

In [26]:
import pandas as pd
import numpy as np

primary = "gpt_excerpt_primary_frame"
secondary = "gpt_excerpt_secondary_frame"

# Normalize frame labels
for col in [primary, secondary]:
    jan6_df[col] = (
        jan6_df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

frame_freq = pd.concat([
    jan6_df[primary].value_counts().rename("Primary N"),
    jan6_df[secondary].value_counts().rename("Secondary N")
], axis=1).fillna(0).astype(int)

frame_freq["Primary %"] = (
    frame_freq["Primary N"] / jan6_df[primary].notna().sum() * 100
)

frame_freq["Secondary %"] = (
    frame_freq["Secondary N"] / len(jan6_df) * 100
)

display(frame_freq.round(1))

,Primary N,Secondary N,Primary %,Secondary %
protester_violence,909,88,62.2,6.0
residual,166,0,11.4,0.0
disruption,118,309,8.1,21.1
impactful,105,72,7.2,4.9
harm_to_protesters,65,61,4.4,4.2
grievance,33,27,2.3,1.8
property_damage,28,68,1.9,4.7
widespread_support,21,9,1.4,0.6
peaceful,16,20,1.1,1.4


# Consistency: how often will a title be delegitimizing while excerpt is legitimizing? What about primary vs secondary frames? 

In [28]:
df = pd.read_csv('full_data.csv')

In [32]:
import pandas as pd

legitimizing = {
    "grievance", "widespread_support", "peaceful",
    "impactful", "harm_to_protesters"
}

delegitimizing = {
    "protester_violence", "property_damage", "disruption"
}

def frame_group(frame):
    if pd.isna(frame) or str(frame).strip() == "":
        return pd.NA
    if frame in legitimizing:
        return "L"
    if frame in delegitimizing:
        return "D"
    if frame == "residual":
        return "N"
    raise ValueError(f"Unknown frame: {frame}")

for col in ["gpt_title_frame", "gpt_excerpt_primary_frame", "gpt_excerpt_secondary_frame"]:
    df[col + "_group"] = df[col].apply(frame_group)

def summarize_pairs(df, col1, col2):
    subset = df[[col1, col2]].dropna().copy()
    subset["pair"] = subset[col1] + " → " + subset[col2]

    counts = subset["pair"].value_counts()
    order = [
        "L → L", "D → D", "L → D", "D → L",
        "N → L", "N → D", "L → N", "D → N", "N → N"
    ]

    result = pd.DataFrame({
        "Count": counts.reindex(order, fill_value=0)
    })
    result["Percent"] = (
        100 * result["Count"] / len(subset)
    ).round(2)

    return result

title_excerpt = summarize_pairs(
    df, "gpt_title_frame_group", "gpt_excerpt_primary_frame_group"
)

primary_secondary = summarize_pairs(
    df, "gpt_excerpt_primary_frame_group", "gpt_excerpt_secondary_frame_group"
)

display(title_excerpt)
display(primary_secondary)

,Count,Percent
pair,,
L → L,2069,22.29
D → D,1496,16.12
L → D,295,3.18
D → L,177,1.91
N → L,2719,29.29
N → D,2177,23.45
L → N,1,0.01
D → N,1,0.01
N → N,348,3.75


,Count,Percent
pair,,
L → L,2278,37.95
D → D,1108,18.46
L → D,958,15.96
D → L,1658,27.62
N → L,0,0.00
N → D,0,0.00
L → N,0,0.00
D → N,0,0.00
N → N,0,0.00


In general, title-excerpt has less L-D and D-L than primary-secondary

The presence of substantive disagreement in primary-secondary justifies allowing for multiple frames

In [34]:

def consistency_summary(df, col1, col2):
    pairs = df[[col1, col2]].dropna()
    a, b = pairs[col1], pairs[col2]

    same = ((a == "L") & (b == "L")) | ((a == "D") & (b == "D"))
    opposing = ((a == "L") & (b == "D")) | ((a == "D") & (b == "L"))
    unclassified = (a == "N") | (b == "N")

    n = len(pairs)
    substantive = (~unclassified).sum()

    return pd.Series({
        "N pairs": n,
        "Same orientation (%)": round(100 * same.sum() / n, 2),
        "Opposing orientation (%)": round(100 * opposing.sum() / n, 2),
        "Unclassified involvement (%)": round(100 * unclassified.sum() / n, 2),
        "Opposing among L/D pairs (%)": round(
            100 * opposing.sum() / substantive, 2
        ) if substantive else float("nan")
    })

summary = pd.DataFrame({
    "Title vs Excerpt": consistency_summary(
        df, "gpt_title_frame_group", "gpt_excerpt_primary_frame_group"
    ),
    "Primary vs Secondary": consistency_summary(
        df, "gpt_excerpt_primary_frame_group", "gpt_excerpt_secondary_frame_group"
    )
}).T

display(summary)


,N pairs,Same orientation (%),Opposing orientation (%),Unclassified involvement (%),Opposing among L/D pairs (%)
Title vs Excerpt,9283.0,38.40,5.08,56.51,11.69
Primary vs Secondary,6002.0,56.41,43.59,0.00,43.59


Same and Opposing Orientation, broken down by topic and political lean

In [39]:
import pandas as pd

# Change these to your actual column names
leaning_col = "political_leaning"
topic_col = "gpt-topic"

def orientation_by_group(df, frame1, frame2, group_col):
    temp = df[[frame1, frame2, group_col]].dropna().copy()

    # Keep only substantive L/D pairs
    temp = temp[
        temp[frame1].isin(["L", "D"]) &
        temp[frame2].isin(["L", "D"])
    ].copy()

    temp["orientation"] = temp.apply(
        lambda r: "Same" if r[frame1] == r[frame2] else "Opposing",
        axis=1
    )

    counts = pd.crosstab(temp[group_col], temp["orientation"])
    counts = counts.reindex(
        columns=["Same", "Opposing"], fill_value=0
    )

    result = counts.copy()
    result["N"] = counts.sum(axis=1)
    result["Same (%)"] = (
        100 * counts["Same"] / result["N"]
    ).round(2)
    result["Opposing (%)"] = (
        100 * counts["Opposing"] / result["N"]
    ).round(2)

    return result[
        ["N", "Same", "Opposing", "Same (%)", "Opposing (%)"]
    ].sort_values("Opposing (%)", ascending=False)


comparisons = {
    "Title vs Excerpt": (
        "gpt_title_frame_group", "gpt_excerpt_primary_frame_group"
    ),
    "Primary vs Secondary": (
        "gpt_excerpt_primary_frame_group", "gpt_excerpt_secondary_frame_group"
    )
}

for name, (frame1, frame2) in comparisons.items():
    print(f"\n=== {name}: Political Leaning ===")
    display(orientation_by_group(df, frame1, frame2, leaning_col))




=== Title vs Excerpt: Political Leaning ===


orientation,N,Same,Opposing,Same (%),Opposing (%)
political_leaning,,,,,
left,205,172,33,83.90,16.10
center,404,352,52,87.13,12.87
left_lean,1945,1714,231,88.12,11.88
right_lean,355,314,41,88.45,11.55
right,1128,1013,115,89.80,10.20



=== Primary vs Secondary: Political Leaning ===


orientation,N,Same,Opposing,Same (%),Opposing (%)
political_leaning,,,,,
right,1577,830,747,52.63,47.37
center,670,372,298,55.52,44.48
right_lean,594,341,253,57.41,42.59
left_lean,2804,1623,1181,57.88,42.12
left,357,220,137,61.62,38.38
